In [5]:
# ═══════════════════════════════════════════════════════════
# IMPORT LIBRARY
# ═══════════════════════════════════════════════════════════
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [6]:
# ═══════════════════════════════════════════════════════════
# KONFIGURASI KOLOM
# ═══════════════════════════════════════════════════════════
KOLOM_HAPUS = [
    'Track URI', 'Release Date', 'Duration (ms)', 'Popularity', 'Explicit',
    'Added By', 'Added At', 'Record Label', 'Key', 'Mode', 'Time Signature'
]

TEXT_COLS = ['Track Name', 'Artist Name(s)', 'Genres']
AUDIO_COLS = ['Danceability', 'Energy', 'Loudness',
              'Speechiness', 'Acousticness', 'Instrumentalness',
              'Liveness', 'Valence', 'Tempo']

In [7]:
# ═══════════════════════════════════════════════════════════
# FUNGSI HELPER
# ═══════════════════════════════════════════════════════════

def case_fold(text):
    if pd.isna(text):
        return ''
    return str(text).lower()

def tokenize(text):
    if pd.isna(text) or not text:
        return []
    text = re.sub(r'[^\w\s]', ' ', str(text))
    tokens = text.split()
    return [t for t in tokens if len(t) > 2]

def tokens_to_str(tokens):
    if not tokens:
        return ''
    return ' '.join(tokens)

In [8]:
# ═══════════════════════════════════════════════════════════
# GENRE NORMALIZATION
# ═════════════════════════════���═════════════════════════════

GENRE_MAPPING = {
    'indorock': 'indonesian_rock',
    'indonesian indie rock': 'indonesian rock',
    'indopop': 'indonesian_pop',
    'indo pop': 'indonesian_pop',
    'indo indie': 'indonesian_indie',
    'malay pop': 'malay',
    'malaysian': 'malay',
}

GENRE_HIERARCHY = {
    'indonesian_rock': ['indonesian indie rock', 'indorock'],
    'indonesian_pop': ['indopop', 'indo pop'],
    'indonesian_indie': ['indo indie'],
    'malay': ['malay pop', 'malaysian'],
}

def clean_genres(genre_str):
    if pd.isna(genre_str) or not genre_str:
        return ''
    genres = [g.strip().lower() for g in str(genre_str).split(',')]
    genres = [g for g in genres if g]
    genres = [GENRE_MAPPING.get(g, g) for g in genres]
    genres = list(dict.fromkeys(genres))
    final_genres = []
    for g in genres:
        is_subset = False
        for main_genre, variants in GENRE_HIERARCHY.items():
            if g in variants and main_genre in genres:
                is_subset = True
                break
        if not is_subset:
            final_genres.append(g)
    return ', '.join(final_genres)

In [9]:
# ═══════════════════════════════════════════════════════════
# 1. DATA CLEANING
# ═══════════════════════════════════════════════════════════

def clean_data(df):
    df_clean = df.drop(columns=KOLOM_HAPUS, errors='ignore')
    df_clean = df_clean.drop_duplicates(subset=['Track Name'], keep='first')
    for col in TEXT_COLS:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('')
    if 'Genres' in df_clean.columns:
        df_clean['Genres'] = df_clean['Genres'].apply(clean_genres)
    for col in AUDIO_COLS:
        if col in df_clean.columns:
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
    df_clean = df_clean.reset_index(drop=True)
    return df_clean

In [10]:
# ═══════════════════════════════════════════════════════════
# 2. CASE FOLDING & TOKENIZATION
# ═══════════════════════════════════════════════════════════════════

def tokenize_data(df_token):
    for col in TEXT_COLS:
        if col in df_token.columns:
            df_token[col] = df_token[col].apply(case_fold)
            df_token[col] = df_token[col].apply(lambda x: tokens_to_str(tokenize(x)))
    df_token['genres_clean'] = df_token['Genres'].apply(
        lambda x: ','.join([g.strip().replace(' ', '_') for g in str(x).split(',') if g.strip()])
    )
    df_token['text_combined'] = df_token['genres_clean'] + ' ' + df_token['Artist Name(s)']
    return df_token

In [11]:
# ═══════════════════════════════════════════════════════════
# 3. NORMALISASI
# ═══════════════════════════════════════════════════════════

def normalize_data(df_norm):
    scaler = MinMaxScaler()
    df_norm[AUDIO_COLS] = scaler.fit_transform(df_norm[AUDIO_COLS])
    return df_norm, scaler

In [12]:
# ═══════════════════════════════════════════════════════════
# JALANKAN PREPROCESSING
# ═══════════════════════════════════════════════════════════

print("\n[1] MEMUAT DATA")
print("-" * 40)
df = pd.read_csv('Song.csv', sep=',', on_bad_lines='skip', encoding='utf-8')
print(f"  Dataset asli : {len(df):,} baris")

print("\n[2] DATA CLEANING")
print("-" * 40)
df_clean = clean_data(df)
print(f"  Setelah cleaning : {len(df_clean):,} baris")
df_clean.to_csv('hasil_01_data_cleaning.csv', index=False)
print("  [OK] hasil_01_data_cleaning.csv")

print("\n[3] CASE FOLDING & TOKENISASI")
print("-" * 40)
df_token = df_clean.copy()
df_token = tokenize_data(df_token)
print("  [OK] Case folding & tokenisasi selesai")
df_token.to_csv('hasil_02_case_folding_tokenisasi.csv', index=False)
print("  [OK] hasil_02_case_folding_tokenisasi.csv")

print("\n[4] NORMALISASI")
print("-" * 40)
df_norm = df_token.copy()
df_norm, scaler = normalize_data(df_norm)
print("  [OK] Normalisasi Min-Max (fitur audio)")
df_norm.to_csv('hasil_03_normalisasi.csv', index=False)
print("  [OK] hasil_03_normalisasi.csv")


[1] MEMUAT DATA
----------------------------------------
  Dataset asli : 11,000 baris

[2] DATA CLEANING
----------------------------------------
  Setelah cleaning : 7,299 baris
  [OK] hasil_01_data_cleaning.csv

[3] CASE FOLDING & TOKENISASI
----------------------------------------
  [OK] Case folding & tokenisasi selesai
  [OK] hasil_02_case_folding_tokenisasi.csv

[4] NORMALISASI
----------------------------------------
  [OK] Normalisasi Min-Max (fitur audio)
  [OK] hasil_03_normalisasi.csv


In [13]:
# Tampilkan hasil preprocessing
df_norm.head()

,Track Name,Album Name,Artist Name(s),Genres,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,genres_clean,text_combined
0,kita usahakan lagi,Kita Usahakan Lagi,batas senja,indonesian pop indonesian indie malay,0.533058,0.549185,0.724574,0.026321,0.220097,0.034444,0.146579,0.454914,0.672487,indonesian_pop_indonesian_indie_malay,indonesian_pop_indonesian_indie_malay batas senja
1,antea,Jalaran Sadrah,barasuara,indonesian indie indonesian rock indonesian po...,0.441116,0.864553,0.827262,0.055959,0.006678,0.000684,0.191712,0.405268,0.622152,indonesian_indie_indonesian_rock_indonesian_po...,indonesian_indie_indonesian_rock_indonesian_po...
2,etalase,Jalaran Sadrah,barasuara,indonesian indie indonesian rock indonesian po...,0.464876,0.940363,0.852142,0.062280,0.001362,0.000033,0.258385,0.468085,0.631407,indonesian_indie_indonesian_rock_indonesian_po...,indonesian_indie_indonesian_rock_indonesian_po...
3,merayakan fana,Jalaran Sadrah,barasuara,indonesian indie indonesian rock indonesian po...,0.273760,0.823111,0.823146,0.049016,0.015372,0.014343,0.338394,0.300912,0.798275,indonesian_indie_indonesian_rock_indonesian_po...,indonesian_indie_indonesian_rock_indonesian_po...
4,habis gelap,Jalaran Sadrah,barasuara,indonesian indie indonesian rock indonesian po...,0.589876,0.661384,0.673539,0.044145,0.047432,0.781818,0.112730,0.430598,0.682429,indonesian_indie_indonesian_rock_indonesian_po...,indonesian_indie_indonesian_rock_indonesian_po...
